In [ ]:
from datetime import datetime as dt
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.reset_defaults()
pd.set_option('display.width', 1000)
colors = {'aws': '#ff9900', 'gcp': '#4285F4'}
pickle_df = pd.read_pickle('../stats-all-latest.pkl')

millis_in_day = 1000 * 60 * 60 * 24
expected_datapoints_in_day = 24 * 12 * 2 * 2 * 3 * 3 * 3 # hours * executions * clouds * images * formats * widths * memory

# First and last executions, timestamp is in millis
min_ts = df["timestamp"].min()
max_ts = df["timestamp"].max()
day_0 = min_ts - min_ts % millis_in_day

print(f'Day 0 start:      {dt.fromtimestamp(day_0/1000)}')
print(f'First timestamp:  {dt.fromtimestamp(min_ts/1000)}')
print(f'Last timestamp:   {dt.fromtimestamp(max_ts/1000)}')
print('')

# start_ts = 1733184000000 # Tue Dec 03 2024 00:00:00 GMT+0000
# end_ts   = 1738540800000 # Mon Feb 03 2025 00:00:00 GMT+0000
start_ts = 1733529600000 # Sat Dec 07 2024 00:00:00 GMT+0000
end_ts   = 1738886400000 # Fri Feb 07 2025 00:00:00 GMT+0000
print(f'Experiment start: {dt.fromtimestamp(start_ts/1000)}')
print(f'Experiment end:   {dt.fromtimestamp(end_ts/1000)}')

# Calculate day of week and hour of day
df = pickle_df.loc[(pickle_df['timestamp'] >= start_ts) & (pickle_df['timestamp'] < end_ts)].copy()
df['day'] = (df['timestamp'] - day_0) // millis_in_day
df['hour'] = (df['timestamp'] - day_0) // (millis_in_day / 24) % 24

# Count occurrences by day
expected_count = 24 * 12 * 2 * 3 * 3 * 3 
# daily_counts = df.groupby(['day', 'providerName', 'imageName', 'outputFormat', 'providerMemoryAssociated', 'outputWidth']) \
daily_counts = df.groupby(['day', 'providerName']).size().reset_index(name='count')
daily_counts['missing_executions'] = expected_count - daily_counts['count']
df_repeated = daily_counts.reindex(daily_counts.index.repeat(daily_counts.missing_executions))
df_transformed = df_repeated.reset_index(drop=True)
print(df_transformed)

g = sns.catplot(
    data=df_transformed,
    kind="count",
    x='day',
    hue='providerName',
    # row='providerName',   # Separate rows for each provider
    palette=colors,       # Use the defined colors
    aspect=2.5,
    height=3.5,
    native_scale=True,
    sharey=False,
)

sns.move_legend(g, loc='upper center', bbox_to_anchor=(0.5, 0), title='', ncol=2)
# plt.suptitle('Missing executions by Day')
plt.suptitle('')
g.set_xlabels('Day')
g.set_ylabels('Count')
plt.tight_layout()
g.savefig(f'../graphs/6-missing-executions-all.pdf', bbox_inches='tight')
plt.show()
plt.close()

# -------------------------

for imageName in ["landscape", "portrait"]:
    df_imageName = df.loc[(df["imageName"] == imageName)]

    expected_count = 24 * 12 # 24 hours * 12 times
    counts = df_imageName.groupby(['day', 'providerName', 'outputFormat', 'outputWidth', 'providerMemoryAssociated']).size().reset_index(name='count')
    counts['missing_executions'] = expected_count-counts['count']
    counts.sort_values(by='missing_executions', inplace=True)

    g = sns.relplot(
        data=counts,
        kind='line',
        x='day',
        y='missing_executions',
        hue='providerName',
        hue_order=['aws', 'gcp'],
        row='outputFormat',                     # Separate rows for each provider
        row_order=['jpeg', 'webp', 'heif'],
        col='providerMemoryAssociated',
        style='outputWidth',
        palette=colors,                         # Use the defined colors
        aspect=1,
        height=3,
    )

    plt.suptitle(f'Missing executions by Day for {imageName}')
    plt.tight_layout()
    sns.move_legend(g, loc='upper center', bbox_to_anchor=(0.5, 0), title='', ncol=8)
    g.set_titles('{row_name}, {col_name}MB')
    g.set_xlabels('Day')
    g.set_ylabels('Count')
    g.savefig(f'../graphs/6-missing-executions-{imageName}.pdf', bbox_inches='tight')
    plt.show()
    plt.close()